In [1]:
# ==============================================================================
# STEP 1: RAPID WORKSPACE DEPENDENCY PROVISIONS
# ==============================================================================
!pip install chromadb
!pip install --no-cache-dir "unsloth[colab-new]" unsloth_zoo -q
!pip install trl peft accelerate bitsandbytes -q
!pip install roslibpy modelscope piper-tts openai-whisper ipywidgets transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [2]:
# Download Piper Nepali TTS Model Asset files
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/ne/ne_NP/google/medium/ne_NP-google-medium.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/ne/ne_NP/google/medium/ne_NP-google-medium.onnx.json

In [15]:
# ==============================================================================
# STEP 2: CLOUD TUNNEL & HARDWARE SYSTEM LINK (CLOUDFLARE TUNNEL INTEGRATION)
# ==============================================================================
import roslibpy
import os
import re
import warnings
from urllib.parse import urlparse
import gc
import json
import wave
import torch
import random
import time
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
from unsloth import FastLanguageModel
from google.colab import userdata
from piper.voice import PiperVoice
from transformers import SeamlessM4Tv2Model, AutoProcessor

warnings.filterwarnings("ignore")
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 🔴 CRITICAL: Paste the exact URL printed by your 'cloudflared tunnel --url http://localhost:9090' terminal here!
TUNNEL_URL = "https://illinois-poster-mins-ask.trycloudflare.com"

# Automatically parse out the clean host domain from the web URL
parsed_url = urlparse(TUNNEL_URL)
TUNNEL_HOST = parsed_url.netloc
TUNNEL_PORT = 443  # Cloudflare Tunnel also proxies over standard HTTPS port 443

print(f"📡 Initializing cloud handshake bridge via Cloudflare Tunnel [{TUNNEL_HOST}]...")
try:
    # Cloudflare Tunnel proxies WSS (Secure WebSockets) over port 443 automatically.
    # Therefore, we set is_secure=True to properly complete the handshake.
    client = roslibpy.Ros(host=TUNNEL_HOST, port=TUNNEL_PORT, is_secure=True)
    client.run(timeout=15)

    # Map to your package's active high-level command/gesture topic
    talker = roslibpy.Topic(client, '/robot/high_level_command', 'std_msgs/msg/String')

    # Verification check
    if client.is_connected:
        print(f"📡 Cloud tunnel link verified active! Connected via Cloudflare Tunnel: {client.is_connected}")
    else:
        raise Exception("Handshake established but websocket reporting inactive state.")

except Exception as e:
    print(f"⚠️ Cloudflare Tunnel Cloud Handshake Failed: {e}")
    print("\n💡 TROUBLESHOOTING TIP:")
    print("1. Ensure 'ros2 launch rosbridge_server rosbridge_websocket_launch.xml' is active on port 9090 locally.")
    print("2. Ensure your local 'cloudflared tunnel --url http://localhost:9090' command is still actively running and hasn't disconnected.")

📡 Initializing cloud handshake bridge via Cloudflare Tunnel [illinois-poster-mins-ask.trycloudflare.com]...
📡 Cloud tunnel link verified active! Connected via Cloudflare Tunnel: True


In [4]:
torch.cuda.empty_cache()

In [5]:
# ==============================================================================
# STEP 3: VRAM-OPTIMIZED MULTI-MODEL INITIALIZATION
# ==============================================================================
# Model A: Unsloth Llama 3.1 LLM (4-Bit Quantized)
import pandas as pd

import ipywidgets as widgets

from IPython.display import display, Audio, clear_output

from unsloth import FastLanguageModel

from google.colab import userdata

from piper.voice import PiperVoice

from transformers import SeamlessM4Tv2Model, AutoProcessor
model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 1024, # Optimized to preserve VRAM
    load_in_4bit = True,
    token = userdata.get('HF_TOKEN')
)
FastLanguageModel.for_inference(model)

# Model B: Translation and Processing Transformers
translator_name = "facebook/hf-seamless-m4t-medium"
processor = AutoProcessor.from_pretrained(translator_name)
translator = SeamlessM4Tv2Model.from_pretrained(translator_name).to("cuda")

# Model C: Piper Audio Synthesis Engine
voice = PiperVoice.load("ne_NP-google-medium.onnx", config_path="ne_NP-google-medium.onnx.json")
print("✅ All AI Processing Units Successfully Staged into GPU Environment.")

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/1211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.84G [00:00<?, ?B/s]

SeamlessM4Tv2Model LOAD REPORT from: facebook/hf-seamless-m4t-medium
Key                                                                               | Status     | 
----------------------------------------------------------------------------------+------------+-
speech_encoder.encoder.layers.{0...11}.conv_module.batch_norm.running_var         | UNEXPECTED | 
speech_encoder.encoder.layers.{0...11}.conv_module.batch_norm.bias                | UNEXPECTED | 
speech_encoder.encoder.layers.{0...11}.self_attn.pos_bias_u                       | UNEXPECTED | 
t2u_model.model.decoder.layers.{0, 1, 2, 3}.cross_attention.out_proj.bias         | UNEXPECTED | 
t2u_model.model.decoder.layers.{0, 1, 2, 3}.ffn.fc1.weight                        | UNEXPECTED | 
speech_encoder.encoder.layers.{0...11}.self_attn.linear_pos.weight                | UNEXPECTED | 
t2u_model.model.decoder.layers.{0, 1, 2, 3}.ffn.fc2.bias                          | UNEXPECTED | 
t2u_model.model.decoder.layers.{0, 1, 2, 3}.ffn.f

generation_config.json:   0%|          | 0.00/5.35k [00:00<?, ?B/s]

✅ All AI Processing Units Successfully Staged into GPU Environment.


In [6]:
# ==============================================================================
# STEP 4: DATASET PARSING & STRUCTURAL INTEGRATION
# ==============================================================================
try:
    df = pd.read_csv("drive/MyDrive/softwarica_knowledge_base_v3.csv")
    df['Topic'] = df['Topic'].astype(str).str.lower()
    print(f"✅ RAG Knowledge Base Loaded. Verified entries: {df['Topic'].count()}")
except Exception as e:
    print(f"⚠️ Knowledge Base file error: {e}. Generating placeholder mock database.")
    df = pd.DataFrame([{
        "Topic": "courses",
        "Information": "Softwarica College offers BSc Hons in Computing and Cyber Security.",
        "Context": "academic",
        "Key Gesture": "hand_side"
    }])

✅ RAG Knowledge Base Loaded. Verified entries: 74


In [7]:
# ==============================================================================
# STEP 5: PERSISTENT CONNECTIVITY MANAGER (PREVENTS REACTOR CRASHES)
# ==============================================================================
import re
import json
import time
import torch
import numpy as np
import chromadb
import roslibpy
from transformers import AutoTokenizer, AutoModel

print("🧠 Loading Sentence-Transformer Embedding Models into GPU...")
eval_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
eval_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to("cuda")

# CRITICAL: Reuse the SAME connection object created in Step 2 — never create a second one!
print("📡 Reusing existing tunnel connection from Step 2...")
global_ros_client = client   # 'client' must already exist from Step 2

if not global_ros_client.is_connected:
    print("⚠️ Step 2 client is not connected. Re-run Step 2 first before continuing.")

current_corridor_width = 2.0

class SoftwaricaRobot:
    def __init__(self, target_topic='/gesture_topic'):
        self.state = "IDLE"
        self.target_topic = target_topic
        self.talker = roslibpy.Topic(global_ros_client, self.target_topic, 'std_msgs/msg/String')
        self.talker.advertise()
        time.sleep(0.3)
        self._was_connected = global_ros_client.is_connected

    def _ensure_advertised(self):
        """Re-advertise if the connection dropped and came back (server lost our advertisement)."""
        global global_ros_client
        currently_connected = global_ros_client.is_connected

        if currently_connected and not self._was_connected:
            # We just reconnected — server no longer knows about our old advertisement
            print("🔄 [RECOVERY]: Connection was restored — re-advertising topic...")
            try:
                self.talker = roslibpy.Topic(global_ros_client, self.target_topic, 'std_msgs/msg/String')
                self.talker.advertise()
                time.sleep(0.3)
            except Exception as e:
                print(f"⚠️ Re-advertise failed: {e}")

        self._was_connected = currently_connected

    def _safe_publish(self, payload):
        global global_ros_client
        try:
            timeout = 0
            while not global_ros_client.is_connected and timeout < 100:
                time.sleep(0.1)
                timeout += 1

            self._ensure_advertised()  # check + recover BEFORE every publish

            if global_ros_client.is_connected:
                self.talker.publish(roslibpy.Message({'data': str(payload)}))
                time.sleep(0.3)
                print(f"   ✓ Confirmed publish attempt for: '{payload}'")
            else:
                print(f"⚠️ Channel never became ready. Message omitted: {payload}")

        except Exception as err:
            print(f"⚠️ Telemetry transmission error: {err}")

    def set_state(self, new_state):
        self.state = new_state.upper()
        print(f"🔄 [STATE CHANGED]: {self.state}")
        self._safe_publish(f"state_{self.state.lower()}")

    def perform_gesture(self, gesture_name):
        clean_name = str(gesture_name).strip().lower().replace(" ", "_")
        valid_gestures = [
            "wave", "hand_up", "hand_down", "hand_shake",
            "hand_side", "walk", "thinking", "walking", "talking"
        ]
        if clean_name not in valid_gestures or clean_name == "none":
            return
        print(f"🤖 [TUNNEL ACTION]: Broadcasting physical gesture payload -> '{clean_name}'")
        self._safe_publish(clean_name)

my_robot = SoftwaricaRobot(target_topic='/robot/high_level_command')

# ==============================================================================
# CHROMADB & DATABASE INGESTION
# ==============================================================================
def get_embedding(text):
    inputs = eval_tokenizer(text, padding=True, truncation=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        model_output = eval_model(**inputs)
    token_embeddings = model_output[0]
    attention_mask = inputs['attention_mask']
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return (sum_embeddings / sum_mask).cpu().numpy().flatten()

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(name="softwarica_kb")
except Exception:
    pass

kb_collection = chroma_client.create_collection(name="softwarica_kb", metadata={"hnsw:space": "cosine"})

print("📦 Ingesting Knowledge Base Array into ChromaDB...")
for idx, row in df.iterrows():
    keywords_string = str(row.get('Topic', '')) + " " + str(row.get('Context', ''))
    factual_text = str(row.get('Information', ''))
    gesture_name = str(row.get('Key Gesture', 'none')).strip().lower()

    dense_vector = get_embedding(keywords_string).tolist()
    kb_collection.add(
        embeddings=[dense_vector],
        documents=[factual_text],
        metadatas=[{"gesture": gesture_name}],
        ids=[f"row_{idx}"]
    )

def query_college_database(search_query):
    global kb_collection
    query_vector = get_embedding(search_query).tolist()
    results = kb_collection.query(query_embeddings=[query_vector], n_results=1)
    if not results or not results['documents'] or len(results['documents'][0]) == 0:
        return {"info": "Softwarica College offers computing degrees.", "gesture": "talking"}
    return {
        "info": results['documents'][0][0],
        "gesture": results['metadatas'][0][0].get('gesture', 'talking')
    }

tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "query_college_database",
            "description": "Queries database for Softwarica College information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "search_query": {"type": "string", "description": "The phrase to query."}
                },
                "required": ["search_query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "trigger_gesture",
            "description": "Triggers a physical hand gesture animation loop.",
            "parameters": {
                "type": "object",
                "properties": {
                    "gesture_name": {
                        "type": "string",
                        "enum": ["wave", "hand_up", "hand_down", "handshake", "hand_side", "walk", "thinking", "walking", "talking"]
                    }
                },
                "required": ["gesture_name"]
            }
        }
    }
]

🧠 Loading Sentence-Transformer Embedding Models into GPU...


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📡 Reusing existing tunnel connection from Step 2...
📦 Ingesting Knowledge Base Array into ChromaDB...


In [8]:
# ==============================================================================
# OPTIMIZED STEP 6: CLEAN INLINE AGENT EXECUTOR
# ==============================================================================
def _softwarica_agent_chat_inner(user_input):
    global current_corridor_width

    # # 1. Enter thinking mode synchronously
    # my_robot.set_state("THINKING")
    # my_robot.perform_gesture("thinking")
    # time.sleep(0.5)  # Quick buffer for Gazebo to parse thinking stance

    system_prompt = (
        "You are an advanced multi-modal autonomous service robot assistant stationed inside Softwarica College.\n"
        "You have access to two tools: 'query_college_database' and 'trigger_gesture'.\n\n"
        "CRITICAL FORCING RULES:\n"
        "1. For general inquiries, you MUST request BOTH tools simultaneously inside your response turn.\n"
        "2. Output your plan inside clean, standard tool-call formatting windows. Do not say conversational words yet."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    inputs = tokenizer.apply_chat_template(messages, tools=tools_definition, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=150, temperature=0.01, use_cache=True)
    raw_response = tokenizer.batch_decode(outputs)[0]

    split_marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    assistant_output = raw_response.split(split_marker)[-1] if split_marker in raw_response else raw_response.split("assistant")[-1]
    assistant_output = assistant_output.replace("<|eot_id|>", "").replace("<|eom_id|>", "").strip()

    extracted_search_query = user_input
    chosen_gesture = "talking"

    # Multi-Call Parser Logic
    tool_calls_found = re.findall(r'name["\s:]+(\w+).*?parameters["\s:]+(\{.*?\} \})', assistant_output, re.DOTALL)

    if not tool_calls_found:
        blocks = re.split(r'(call:\s*\w+|"name"\s*:\s*"\w+")', assistant_output)
        current_name = None
        for block in blocks:
            cleaned_block = block.strip()
            if "query_college_database" in cleaned_block:
                current_name = "query_college_database"
            elif "trigger_gesture" in cleaned_block:
                current_name = "trigger_gesture"
            elif cleaned_block.startswith("{") and current_name:
                try:
                    parsed_args = json.loads(cleaned_block)
                    if current_name == "query_college_database":
                        extracted_search_query = parsed_args.get("search_query", user_input)
                    elif current_name == "trigger_gesture":
                        chosen_gesture = parsed_args.get("gesture_name", "talking")
                except Exception:
                    pass
                current_name = None
    else:
        for name, args_str in tool_calls_found:
            try:
                parsed_args = json.loads(args_str)
                if name == "query_college_database":
                    extracted_search_query = parsed_args.get("search_query", user_input)
                elif name == "trigger_gesture":
                    chosen_gesture = parsed_args.get("gesture_name", "talking")
            except Exception:
                pass

    # Direct String Fallback Matches to support local command configurations
    if any(w in user_input.lower() for w in ["side", "to the side"]):
        chosen_gesture = "hand_side"
    elif any(w in user_input.lower() for w in ["hands up", "raise your hand", "put your hand up"]):
        chosen_gesture = "hand_up"
    elif any(w in user_input.lower() for w in ["shake hands", "handshake"]):
        chosen_gesture = "hand_shake"
    elif any(w in user_input.lower() for w in ["walk", "move forward", "move", "walk ahead"]):
        chosen_gesture = "walking"

    db_result = query_college_database(extracted_search_query)
    factual_context = db_result.get("info", "")

    if db_result.get("gesture") != "none" and chosen_gesture == "talking":
        chosen_gesture = db_result.get("gesture")

    # --- SYNTHESIZE SPEECH TEXT ---
    rag_messages = [
        {
            "role": "system",
            "content": (
                "You are the vocal module of the Softwarica College service robot.\n"
                f"FACT BLOCK: {factual_context}\n\n"
                "INSTRUCTIONS:\n"
                "1. Answer the query based on the fact block context.\n"
                "2. If the context is missing, output factual information regarding Softwarica College anyway.\n"
                "3. Keep your output short, professional, and limited to 1-2 sentences maximum."
            )
        },
        {"role": "user", "content": user_input}
    ]

    rag_inputs = tokenizer.apply_chat_template(rag_messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    rag_outputs = model.generate(input_ids=rag_inputs, max_new_tokens=80, temperature=0.1, use_cache=True)
    rag_response = tokenizer.batch_decode(rag_outputs)[0]

    speech_text = rag_response.split(split_marker)[-1] if split_marker in rag_response else rag_response.split("assistant")[-1]
    speech_text = speech_text.replace("<|eot_id|>", "").replace("<|eom_id|>", "").strip()
    speech_text = re.sub(r'(<.*?>|\{.*?\}|\[.*?\])', '', speech_text).strip()

    # Pass BOTH the spoken text and the command string back out to the UI scheduler
    return speech_text, chosen_gesture


# ==============================================================================
# OVERHAULED STEP 7: PIPELINE TIMING OVERRIDE CONTROLLER
# ==============================================================================
def process_interaction(b):
    global current_corridor_width
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''

    # We wrap the logic cleanly within output_area contexts to track logging order
    with output_area:
        print(f"\n👤 User: {user_msg}")
        print(f"📏 Current Corridor Profile: {current_corridor_width:.2f} meters")

        # 1. Extract concept and targeted execution command
        reply, chosen_gesture = _softwarica_agent_chat_inner(user_msg)
        print(f"🤖 Robot (English Concept): {reply}")

        # 2. Transition state to speaking and instantly transmit the gesture payload
        my_robot.set_state("SPEAKING")
        time.sleep(0.2) # Clear state buffer loop

        my_robot.perform_gesture(chosen_gesture)

        # --- CRITICAL SOCKET FLUSH BUFFER ---
        # Forces Python to yield control to the roslibpy network worker thread
        # so it can push bytes across the tunnel BEFORE the heavy TTS block locks the CPU
        time.sleep(0.6)

    # 3. Handle speech audio generation safely now that network packets have cleared
    with audio_area:
        clear_output(wait=True)
        audio_widget = robot_respond(reply)
        if audio_widget:
            display(audio_widget)

    # 4. HARDWARE LOCK: Hold state channel open based on action parameters
    with output_area:
        if chosen_gesture in ["walking", "hand_shake"]:
            print(f"⏳ [TIMING LOCK]: Holding state channel for complex motion trajectory ({chosen_gesture})...")
            time.sleep(6.5)
        else:
            print(f"⏳ [TIMING LOCK]: Holding state channel for quick gesture ({chosen_gesture})...")
            time.sleep(3.5)

        # 5. Safely reset the channel state to IDLE
        my_robot.set_state("IDLE")

In [9]:
# ==============================================================================
# FIXED STEP 7.5: COLAB INTERACTIVE USER INTERFACE FRAMEWORK (DYNAMIC MONITORING)
# ==============================================================================
import torch
import wave
import time
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio

def robot_respond(english_text):
    try:
        print("🌐 Executing SeamlessM4T English to Nepali Translation...")
        inputs = processor(text=english_text, src_lang="eng", return_tensors="pt").to("cuda")

        with torch.no_grad():
            tokens = translator.generate(**inputs, tgt_lang="npi", generate_speech=False)

        nepali_text = processor.decode(tokens[0].tolist()[0], skip_special_tokens=True)
        print(f"🇳🇵 Nepali Text: {nepali_text}")

        output_file = "robot_speech.wav"
        audio_bytes = b"".join([chunk.audio_int16_bytes for chunk in voice.synthesize(nepali_text)])

        with wave.open(output_file, "wb") as wav:
            wav.setnchannels(1)
            wav.setsampwidth(2)
            wav.setframerate(22050)
            wav.writeframes(audio_bytes)

        return Audio(output_file, autoplay=True)

    except Exception as audio_err:
        print(f"⚠️ Multimedia synthesis pipeline error: {audio_err}")
        return None

def get_audio_duration(filename="robot_speech.wav"):
    try:
        with wave.open(filename, 'r') as f:
            frames = f.getnframes()
            rate = f.getframerate()
            return frames / float(rate)
    except Exception:
        return 3.5 # Fallback baseline duration if file doesn't exist

# Cleaned, non-blocking single interaction callback
def process_interaction(b):
    global current_corridor_width
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''

    with output_area:
        print(f"\n👤 User: {user_msg}")

        # 1. Compute high-level intent text and gesture tokens
        reply, chosen_gesture = _softwarica_agent_chat_inner(user_msg)
        print(f"🤖 Robot (English Concept): {reply}")

        # 2. Update state to SPEAKING and instantly deploy the gesture payload
        # Doing this BEFORE generating audio prevents CPU starvation on the network socket!
        my_robot.set_state("SPEAKING")
        time.sleep(0.1) # Brief buffer to clear state payload

        print(f"🤖 [TUNNEL ACTION]: Broadcasting physical gesture payload -> '{chosen_gesture}'")
        my_robot.perform_gesture(chosen_gesture)

        # 3. --- CRITICAL SOCKET FLUSH BUFFER ---
        # Yields execution control so roslibpy can safely push the bytes over
        # the network tunnel before the heavy TTS pipeline monopolizes the hardware
        time.sleep(0.8)

    # 4. Generate audio waves now that the network pipeline is clear
    with audio_area:
        clear_output(wait=True)
        audio_widget = robot_respond(reply)
        if audio_widget:
            display(audio_widget)

    with output_area:
        # 5. Read the true play duration of the audio asset file
        speech_duration = get_audio_duration("robot_speech.wav")
        print(f"🔊 Synthesized Speech Duration: {speech_duration:.2f} seconds")

        # 6. DYNAMIC HARDWARE LOCK: Hold state channel open for the full duration of speech
        if chosen_gesture in ["walking", "hand_shake"]:
            execution_window = max(speech_duration, 6.5)
        else:
            execution_window = speech_duration

        print(f"⏳ [TIMING LOCK]: Holding posture loop for {execution_window:.2f} seconds...")
        time.sleep(execution_window)

        # 7. Safely return state to IDLE
        my_robot.set_state("IDLE")

# --- UI Layout Initialization ---
text_input = widgets.Text(
    placeholder='Ask the Softwarica Robot...',
    description='Input:',
    layout=widgets.Layout(width='75%')
)
send_btn = widgets.Button(description='Send Payload', button_style='success', icon='send')
output_area = widgets.Output()
audio_area = widgets.Output()

send_btn.on_click(process_interaction)
display(widgets.VBox([widgets.HBox([text_input, send_btn]), output_area, audio_area]))
print("🚀 Execution UI Matrix Rendered. System Listening via Tunnel Intercept Workflow.")

🚀 Execution UI Matrix Rendered. System Listening via Tunnel Intercept Workflow.


# TEST

In [16]:
# ==============================================================================
# STEP 8: COMPREHENSIVE MULTI-MODAL EVALUATION SUITE
# ==============================================================================
import time
import numpy as np
import torch

def run_system_evaluation(test_dataset):
    """
    Evaluates the RAG LLM response quality, gesture mapping correctness,
    hallucination frequencies, and operational latency boundaries.
    """
    semantic_scores = []
    gesture_matches = 0
    hallucination_count = 0
    latencies = []

    print("🧪 [EVALUATION] Starting Comprehensive Multi-Modal Verification Battery...\n")
    print("-" * 90)

    for i, (query, gold_text, expected_gesture) in enumerate(test_dataset, 1):
        print(f"📋 Test Case {i}: '{query}'")

        # 1. Measure Latency Performance
        start_time = time.perf_counter()
        try:
            # Safely unpack the inner tuple output without crashing
            generated_reply, predicted_gesture = _softwarica_agent_chat_inner(query)
            end_time = time.perf_counter()

            latency_ms = (end_time - start_time) * 1000
            latencies.append(latency_ms)

        except Exception as e:
            print(f"   ❌ Execution Failed: {e}\n")
            continue

        # 2. Compute Semantic Cosine Similarity via Embedding Engine
        emb_gen = get_embedding(generated_reply)
        emb_gold = get_embedding(gold_text)

        dot_product = np.dot(emb_gen, emb_gold)
        norm_gen = np.linalg.norm(emb_gen)
        norm_gold = np.linalg.norm(emb_gold)

        similarity = dot_product / (norm_gen * norm_gold) if (norm_gen * norm_gold) > 0 else 0
        score = max(0.0, min(1.0, float(similarity)))
        semantic_scores.append(score)

        # 3. Assess Gesture Selection Classification
        gesture_match = (str(predicted_gesture).strip().lower() == str(expected_gesture).strip().lower())
        if gesture_match:
            gesture_matches += 1

        # 4. Check for Systemic Meta-Commentary Leakage / Hallucinations
        meta_phrases = ["database", "retrieved", "tool", "context", "error", "exception", "call:", "parameters"]
        has_hallucination = any(phrase in generated_reply.lower() for phrase in meta_phrases)
        if has_hallucination:
            hallucination_count += 1

        # Log Individual Turn Performance Summary
        print(f"   ↳ 🤖 Generated Text : \"{generated_reply}\"")
        print(f"   ↳ 🎭 Gesture Match  : {predicted_gesture} (Expected: {expected_gesture}) ➔ {'✅ MATCH' if gesture_match else '❌ MISMATCH'}")
        print(f"   ↳ 📊 Semantic Match : {score * 100:.2f}%")
        print(f"   ↳ ⚡ Compute Latency : {latency_ms:.2f} ms")
        if has_hallucination:
            print("   ⚠️  [ALERT]: Meta-commentary or structural JSON leaked into speech text!")
        print("-" * 90)

    # Calculate Macro Aggregations for Academic Reporting
    total_cases = len(semantic_scores)
    if total_cases == 0:
        print("❌ No test assertions processed successfully.")
        return

    mean_semantic_accuracy = np.mean(semantic_scores) * 100
    gesture_accuracy = (gesture_matches / total_cases) * 100
    hallucination_rate = (hallucination_count / total_cases) * 100
    avg_latency = np.mean(latencies)

    # ==========================================================================
    # ACADEMIC METRICS DASHBOARD OUTPUT
    # ==========================================================================
    print("\n📊 " + "="*25 + " FINAL ACADEMIC METRICS DASHBOARD " + "="*25)
    print(f"✅ Mean Semantic Context Accuracy         : {mean_semantic_accuracy:.2f}%")
    print(f"🤖 Gesture Selection Accuracy             : {gesture_accuracy:.2f}% ({gesture_matches}/{total_cases} Correct)")
    print(f"⚠️  Meta-Commentary/Hallucination Rate    : {hallucination_rate:.2f}% ({hallucination_count} Leaks)")
    print(f"⚡ Average Pipeline Inference Latency     : {avg_latency:.2f} ms")
    print("=" * 84)

    # Validation Safeguard Thresholds for Hardware Readiness
    if mean_semantic_accuracy >= 75.0 and gesture_accuracy >= 66.0 and hallucination_rate <= 20.0:
        print("🚀 [STATUS]: Pipeline meets architectural standards for hardware initialization.")
    else:
        print("⚠️  [STATUS]: Weak validation performance detected. Fine-tune RAG prompts before deployment.")

# ==============================================================================
# EXTENDED VALIDATION GROUND-TRUTH DATASET (Softwarica College Specific)
# ==============================================================================
# Schema: (User Prompt, Expected Factual Gold Text, Expected Gesture Output Token)
academic_test_battery = [
    (
        "what courses do you have?",
        "Softwarica College offers BSc Hons in Computing and Cyber Security.",
        "talking"
    ),
    (
        "tell me about the fees",
        "The admission and semester cost structure for computing courses incorporates registration charges and standard semester fees.",
        "talking"
    ),
    (
        "where is the campus located?",
        "Softwarica College is located at Mahakavi Marg, Dillibazar, Kathmandu.",
        "talking"
    ),
    (
        "can you walk forward?",
        "Initiating motor control sequences to execute a linear walking trajectory.",
        "walking"
    ),
    (
        "shake hands with me",
        "Extending mechanical arm to perform a formal greeting handshake.",
        "hand_shake"
    )
]

# Run the unified evaluation loop over your active system architecture
run_system_evaluation(academic_test_battery)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 [EVALUATION] Starting Comprehensive Multi-Modal Verification Battery...

------------------------------------------------------------------------------------------
📋 Test Case 1: 'what courses do you have?'


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   ↳ 🤖 Generated Text : "Softwarica College offers four programs, including three undergraduate BSc Honours degrees and one postgraduate MSc degree."
   ↳ 🎭 Gesture Match  : talking (Expected: talking) ➔ ✅ MATCH
   ↳ 📊 Semantic Match : 72.16%
   ↳ ⚡ Compute Latency : 4977.75 ms
------------------------------------------------------------------------------------------
📋 Test Case 2: 'tell me about the fees'


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

   ↳ 🤖 Generated Text : "Softwarica College is located in Kathmandu, Nepal."
   ↳ 🎭 Gesture Match  : talking (Expected: talking) ➔ ✅ MATCH
   ↳ 📊 Semantic Match : 92.68%
   ↳ ⚡ Compute Latency : 2581.23 ms
------------------------------------------------------------------------------------------
📋 Test Case 4: 'can you walk forward?'


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   ↳ 🤖 Generated Text : "Yes, I can perform a walking gesture on request. Please say "walk forward" to initiate the action."
   ↳ 🎭 Gesture Match  : walking (Expected: walking) ➔ ✅ MATCH
   ↳ 📊 Semantic Match : 44.93%
   ↳ ⚡ Compute Latency : 2908.82 ms
------------------------------------------------------------------------------------------
📋 Test Case 5: 'shake hands with me'


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   ↳ 🤖 Generated Text : "I'm happy to meet you, but as a vocal module, I don't have a physical hand to shake. However, I'm here to assist you with any questions or information you may need about Softwarica College."
   ↳ 🎭 Gesture Match  : hand_shake (Expected: hand_shake) ➔ ✅ MATCH
   ↳ 📊 Semantic Match : 37.57%
   ↳ ⚡ Compute Latency : 5751.98 ms
------------------------------------------------------------------------------------------

📊 ========================= FINAL ACADEMIC METRICS DASHBOARD =========================
✅ Mean Semantic Context Accuracy         : 58.97%
🤖 Gesture Selection Accuracy             : 80.00% (4/5 Correct)
⚠️  Meta-Commentary/Hallucination Rate    : 0.00% (0 Leaks)
⚡ Average Pipeline Inference Latency     : 4080.67 ms
⚠️  [STATUS]: Weak validation performance detected. Fine-tune RAG prompts before deployment.
